# Tutorial 13 — Reward Model Training

**Series:** Training Language Models from Scratch: A Hacker's Guide  
**Part IV — Alignment**  
**Follows:** Tutorial 12 (Direct Preference Optimization)  
**Precedes:** Tutorial 14 (GRPO)

---

## What This Tutorial Covers

A reward model (RM) is a neural network that takes a prompt and a response
and outputs a scalar: how good is this response? It is the bridge between
human preferences — which are expensive to collect — and the training signal
that GRPO (Tutorial 14) needs on every step.

Everything in this tutorial is supervised learning. No RL, no policy
gradients, no sampling. Just a model, a loss function, and a dataset of
human preference pairs.

Topics:

1. **The Bradley-Terry model** — the probability model underlying reward
   training. Why it looks exactly like logistic regression.
2. **Architecture** — how to turn a language model into a scalar scorer.
   The reward head. Why we reuse the pretrained backbone.
3. **The ranking loss** — margin loss vs cross-entropy loss for preferences.
   When each is appropriate.
4. **The `RewardDataset`** — same preference pairs as DPO, different targets.
5. **Training the reward model** — the loop, the metrics, calibration.
6. **Evaluating the reward model** — accuracy, reward distribution,
   the length bias problem and how to detect it.
7. **Using the reward model** — scoring responses at inference time,
   best-of-N sampling as a cheap alternative to full RLHF.

---

## 1. The Bradley-Terry Model

In Tutorial 12 we saw the Bradley-Terry model stated briefly:

$$P(y_w \succ y_l \mid x) = \sigma(r(x, y_w) - r(x, y_l))$$

Let's understand it properly, because it is the entire theoretical foundation
of reward model training.

Suppose each response $y$ has a latent "quality" score $r(x, y)$. We model
the probability that a human prefers $y_w$ over $y_l$ as proportional to
the difference in their scores, passed through a sigmoid. When scores are
equal, preference probability is 0.5 — a coin flip. As the margin grows,
the probability of preferring the better response approaches 1.

This is exactly **logistic regression**[^bt_lr] with features
$(r(x, y_w) - r(x, y_l))$ and label 1

[^bt_lr]: The Bradley-Terry model is logistic regression on the reward gap. This means reward model training has the same convergence guarantees as logistic regression — in particular, the loss is convex in the linear weights (though not in the full network). The optimal solution assigns reward differences proportional to log-odds of human preference. (the human chose $y_w$).

The training objective is maximum likelihood — maximize the probability
of the observed preferences:

$$\mathcal{L}_{\text{RM}} = -\mathbb{E}_{(x, y_w, y_l)}\left[\log \sigma\!\left(r(x, y_w) - r(x, y_l)\right)\right]$$

This is equivalent to binary cross-entropy where the target is always 1
(chosen is always preferred) and the logit is the reward margin
$r(x, y_w) - r(x, y_l)$.

The reward model learns to assign higher scores to chosen responses and
lower scores to rejected responses — not by being told absolute values,
but purely through the relative signal of pairwise preferences.

---

## 2. Architecture: Language Model + Scalar Head

A reward model is a pretrained language model with the language modeling
head removed and replaced with a linear layer that outputs a single scalar.

```
Input:  [prompt tokens] + [response tokens]
         ↓
Transformer backbone (frozen or fine-tuned)
         ↓
Final hidden state at the last token position: h ∈ ℝ^d_model
         ↓
Linear head: W ∈ ℝ^{d_model × 1}
         ↓
Output: scalar reward r ∈ ℝ
```

[**Why the last token?** The causal attention mask means the last token's
hidden state has attended to every preceding token]{.mark} — it is the only position
that has seen the full sequence. It is the natural "summary" representation
of the entire (prompt, response) pair.

**Why reuse the pretrained backbone?** The backbone already knows how to
represent text. Training a reward model from scratch would require the model
to simultaneously learn language understanding and preference scoring.
Reusing the backbone means we only need to learn the scalar projection on
top of already-rich representations. [This is why reward models can be trained
on relatively small preference datasets (10K–100K pairs).]{.mark}

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

from tutorial_02 import GPT, NanoGPTConfig


class RewardModel(nn.Module):
    """
    A language model backbone with a scalar reward head.

    The backbone is initialized from a pretrained (SFT) model.
    The reward head is a single linear layer initialized to zero
    (so initial rewards are near zero for all inputs — neutral baseline).

    Forward pass returns a scalar reward per sequence in the batch.
    """

    def __init__(self, config: NanoGPTConfig, backbone_path: str = None):
        super().__init__()
        self.backbone = GPT(config)

        # Remove the language modeling head — we don't need it
        # The backbone's lm_head is replaced by our reward head
        d_model = config.d_model
        self.reward_head = nn.Linear(d_model, 1, bias=False)

        # Initialize reward head to zero — neutral initial rewards
        nn.init.zeros_(self.reward_head.weight)

        # Load pretrained backbone if provided
        if backbone_path is not None:
            ckpt = torch.load(backbone_path, map_location='cpu')
            state = ckpt['model'] if 'model' in ckpt else ckpt
            # Load only the backbone weights, not the lm_head
            backbone_state = {k: v for k, v in state.items()
                              if not k.startswith('lm_head')}
            missing, unexpected = self.backbone.load_state_dict(
                backbone_state, strict=False
            )
            print(f"Loaded backbone: {len(backbone_state)} tensors")
            if unexpected:
                print(f"  Unexpected keys: {unexpected[:3]}")

    def forward(
        self,
        input_ids:      torch.Tensor,   # (B, T)
        attention_mask: torch.Tensor = None,  # (B, T), 1=real token, 0=pad
    ) -> torch.Tensor:
        """
        Returns scalar reward for each sequence.
        Shape: (B,)
        """
        # Get hidden states from the backbone
        # We need all hidden states, not just the final logits
        hidden_states = self.backbone.get_hidden_states(input_ids)
        # hidden_states: (B, T, d_model)

        # Extract the hidden state at the last *real* token position
        # (not the padding position, if any)
        if attention_mask is not None:
            # Find the last non-padding position for each sequence
            # sequence_lengths[i] = index of last real token
            sequence_lengths = attention_mask.sum(dim=1) - 1   # (B,)
        else:
            sequence_lengths = torch.full(
                (input_ids.size(0),), input_ids.size(1) - 1,
                dtype=torch.long, device=input_ids.device
            )

        # Gather the hidden state at the last real token
        batch_size = input_ids.size(0)
        last_hidden = hidden_states[
            torch.arange(batch_size, device=input_ids.device),
            sequence_lengths,
        ]   # (B, d_model)

        # Project to scalar reward
        reward = self.reward_head(last_hidden).squeeze(-1)   # (B,)
        return reward

### Adding `get_hidden_states` to the GPT backbone

The `GPT` class from Tutorial 2 returns `(logits, loss)`. For the reward
model we need the hidden states before the LM head. Add this method:

In [ ]:
# In tutorial_02.py — add to the GPT class:

def get_hidden_states(self, idx: torch.Tensor) -> torch.Tensor:
    """
    Forward pass returning hidden states (before lm_head).
    Shape: (B, T, d_model)
    """
    B, T = idx.shape
    assert T <= self.config.max_seq_len

    pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
    tok_emb = self.token_embedding(idx)        # (B, T, d_model)
    pos_emb = self.position_embedding(pos)     # (T, d_model)  [or RoPE]
    x = self.dropout(tok_emb + pos_emb)

    for block in self.blocks:
        x = block(x)

    x = self.ln_f(x)   # final layer norm
    return x            # (B, T, d_model) — hidden states, not logits

---

## 3. The Reward Loss

The ranking loss for a single preference pair $(y_w, y_l)$:

$$\mathcal{L} = -\log \sigma(r_w - r_l) = \log(1 + e^{-(r_w - r_l)})$$

This is the softplus of the negative margin. It is zero when $r_w \gg r_l$
(model correctly ranks the pair with large margin) and approaches $\log 2$
when $r_w = r_l$ (model is indifferent).

In [ ]:
def reward_loss(
    chosen_rewards:   torch.Tensor,   # (B,)
    rejected_rewards: torch.Tensor,   # (B,)
    margin:           float = 0.0,
) -> tuple[torch.Tensor, dict]:
    """
    Bradley-Terry ranking loss with optional margin.

    margin > 0 requires the reward gap to exceed `margin` before the
    loss reaches zero. This prevents the model from finding trivially
    small separations and encourages larger, more confident margins.

    Returns (loss, metrics_dict).
    """
    reward_gap = chosen_rewards - rejected_rewards   # (B,)

    if margin > 0:
        # Margin loss: only reward gaps > margin score zero loss
        loss = -F.logsigmoid(reward_gap - margin).mean()
    else:
        loss = -F.logsigmoid(reward_gap).mean()

    metrics = {
        'loss':             loss.item(),
        'reward_gap':       reward_gap.mean().item(),
        'chosen_reward':    chosen_rewards.mean().item(),
        'rejected_reward':  rejected_rewards.mean().item(),
        'accuracy':         (reward_gap > 0).float().mean().item(),
    }
    return loss, metrics

[**Margin loss vs standard loss:** Without a margin,]{.underline} the loss approaches
zero whenever $r_w > r_l$ by any amount — including $10^{-6}$. The model
can satisfy the training objective with tiny, fragile separations. A margin
of 0.5–1.0 forces the model to produce confident, well-separated scores.
This makes the reward model more useful as a ranking signal downstream.

---

## 4. The Reward Dataset

The same preference pairs from Tutorial 12, but we return rewards rather
than DPO log-prob targets:

In [ ]:
import json
from torch.utils.data import Dataset
from pathlib import Path

class RewardDataset(Dataset):
    """
    Loads preference pairs and returns tokenized (chosen, rejected) pairs.

    Each sample returns:
    - chosen_ids:    (T_c,) token IDs of [prompt + chosen response]
    - rejected_ids:  (T_r,) token IDs of [prompt + rejected response]
    - attention_mask_chosen:   (T_c,) 1 for real tokens
    - attention_mask_rejected: (T_r,) 1 for real tokens
    """

    def __init__(self, data_path: str, tokenizer, max_length: int = 512):
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.samples    = []

        with open(data_path) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    self.samples.append(json.loads(line))
                except json.JSONDecodeError:
                    continue

        print(f"RewardDataset: {len(self.samples)} preference pairs")

    def _encode(self, prompt: str, response: str):
        text   = (f"{SPECIAL_TOKENS['user']}\n{prompt.strip()}\n"
                  f"{SPECIAL_TOKENS['end']}\n"
                  f"{SPECIAL_TOKENS['assistant']}\n"
                  f"{response.strip()}\n"
                  f"{SPECIAL_TOKENS['end']}")
        tokens = self.tokenizer.encode(text)[:self.max_length]
        ids    = torch.tensor(tokens, dtype=torch.long)
        mask   = torch.ones_like(ids)
        return ids, mask

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        c_ids, c_mask = self._encode(s['prompt'], s['chosen'])
        r_ids, r_mask = self._encode(s['prompt'], s['rejected'])
        return c_ids, c_mask, r_ids, r_mask


def collate_reward(batch):
    """Pad chosen and rejected sequences separately to their own max lengths."""
    c_ids, c_masks, r_ids, r_masks = zip(*batch)

    def pad(tensors, pad_value=0):
        max_len = max(t.size(0) for t in tensors)
        out = torch.full((len(tensors), max_len), pad_value, dtype=torch.long)
        for i, t in enumerate(tensors):
            out[i, :t.size(0)] = t
        return out

    return (
        pad(c_ids),   pad(c_masks),
        pad(r_ids),   pad(r_masks),
    )

---

## 5. The Training Loop

In [ ]:
import numpy as np
import time
from torch.utils.data import DataLoader, random_split

def train_reward_model(
    backbone_path: str,
    data_path:     str,
    output_dir:    str,
    max_lr:        float = 1e-4,
    min_lr:        float = 1e-5,
    warmup_steps:  int   = 50,
    max_steps:     int   = 1000,
    batch_size:    int   = 4,
    max_length:    int   = 256,
    margin:        float = 0.5,
    eval_every:    int   = 100,
    freeze_layers: int   = 4,    # freeze first N transformer blocks
):
    """
    Train a reward model on preference pairs.

    freeze_layers: freeze the first N transformer blocks of the backbone.
    Fine-tuning the full backbone on a small preference dataset risks
    overfitting. Freezing early layers is a good regularizer.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    dtype  = torch.bfloat16 if device.type == 'cuda' else torch.float32
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    from tutorial_03 import Tokenizer
    tok = Tokenizer.load('nano_tokenizer.json')

    # ---- Model ----
    config = NanoGPTConfig()
    model  = RewardModel(config, backbone_path=backbone_path).to(device)

    # Freeze early backbone layers
    for i, block in enumerate(model.backbone.blocks):
        if i < freeze_layers:
            for p in block.parameters():
                p.requires_grad_(False)

    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total     = sum(p.numel() for p in model.parameters())
    print(f"Trainable: {n_trainable:,} / {n_total:,} "
          f"({100*n_trainable/n_total:.1f}%)")

    # ---- Data ----
    ds       = RewardDataset(data_path, tok, max_length=max_length)
    val_size = max(1, len(ds) // 10)
    train_ds, val_ds = random_split(ds, [len(ds) - val_size, val_size])

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              collate_fn=collate_reward, num_workers=2,
                              pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              collate_fn=collate_reward, num_workers=1)

    # ---- Optimizer — separate LR for head vs backbone ----
    # The reward head starts at zero and needs to learn from scratch.
    # The backbone is pretrained and should move slowly.
    optimizer = torch.optim.AdamW([
        {'params': model.reward_head.parameters(),  'lr': max_lr * 10},
        {'params': [p for n, p in model.backbone.named_parameters()
                    if p.requires_grad],             'lr': max_lr},
    ], weight_decay=0.01)

    scheduler = make_cosine_schedule(optimizer, max_lr, min_lr,
                                      warmup_steps, max_steps)

    # ---- Training loop ----
    model.train()
    train_iter = iter(train_loader)
    history    = []
    best_acc   = 0.0

    for step in range(max_steps):
        try:
            c_ids, c_mask, r_ids, r_mask = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            c_ids, c_mask, r_ids, r_mask = next(train_iter)

        c_ids  = c_ids.to(device);  c_mask = c_mask.to(device)
        r_ids  = r_ids.to(device);  r_mask = r_mask.to(device)

        with torch.autocast(device_type=device.type, dtype=dtype):
            chosen_r   = model(c_ids, c_mask)
            rejected_r = model(r_ids, r_mask)
            loss, metrics = reward_loss(chosen_r, rejected_r, margin=margin)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        history.append(metrics)

        if step % 50 == 0:
            lr = optimizer.param_groups[0]['lr']
            print(
                f"step {step:4d}  "
                f"loss={metrics['loss']:.4f}  "
                f"gap={metrics['reward_gap']:.3f}  "
                f"acc={metrics['accuracy']:.2%}  "
                f"lr={lr:.2e}"
            )

        # ---- Eval ----
        if step % eval_every == 0 and step > 0:
            model.eval()
            eval_metrics = []
            with torch.no_grad():
                for c_ids_v, c_mask_v, r_ids_v, r_mask_v in val_loader:
                    c_ids_v  = c_ids_v.to(device);  c_mask_v = c_mask_v.to(device)
                    r_ids_v  = r_ids_v.to(device);  r_mask_v = r_mask_v.to(device)
                    with torch.autocast(device_type=device.type, dtype=dtype):
                        cr = model(c_ids_v, c_mask_v)
                        rr = model(r_ids_v, r_mask_v)
                        _, m = reward_loss(cr, rr, margin=0.0)
                    eval_metrics.append(m)
            model.train()

            avg_acc = np.mean([m['accuracy'] for m in eval_metrics])
            avg_gap = np.mean([m['reward_gap'] for m in eval_metrics])
            print(f"  [eval] acc={avg_acc:.2%}  gap={avg_gap:.3f}")

            if avg_acc > best_acc:
                best_acc = avg_acc
                torch.save({
                    'step':    step,
                    'model':   model.state_dict(),
                    'config':  config,
                    'acc':     best_acc,
                    'gap':     avg_gap,
                }, f'{output_dir}/reward_model_best.pt')
                print(f"  ✓ New best: acc={best_acc:.2%}")

    print(f"\nReward model training complete. Best acc: {best_acc:.2%}")
    return model, history

---

## 6. Evaluating the Reward Model

Accuracy (fraction of pairs correctly ranked) is the primary metric, but
three additional checks are essential before using the RM downstream:

### Reward distribution

The reward model should assign scores with reasonable spread. [If all scores
cluster near zero, the model has not learned to discriminate.]{.underline} If scores span
$[-100, 100]$, the model is overconfident and will dominate the KL term in
GRPO.

In [ ]:
@torch.no_grad()
def analyze_reward_distribution(
    model:      RewardModel,
    dataloader: DataLoader,
    device:     torch.device,
    n_batches:  int = 50,
) -> dict:
    """
    Compute statistics of the reward distribution over a dataset.
    Returns mean, std, min, max, and a histogram of scores.
    """
    model.eval()
    all_chosen   = []
    all_rejected = []

    for i, (c_ids, c_mask, r_ids, r_mask) in enumerate(dataloader):
        if i >= n_batches:
            break
        c_ids = c_ids.to(device); c_mask = c_mask.to(device)
        r_ids = r_ids.to(device); r_mask = r_mask.to(device)

        cr = model(c_ids, c_mask).cpu().float()
        rr = model(r_ids, r_mask).cpu().float()
        all_chosen.extend(cr.tolist())
        all_rejected.extend(rr.tolist())

    chosen   = np.array(all_chosen)
    rejected = np.array(all_rejected)
    all_r    = np.concatenate([chosen, rejected])

    stats = {
        'chosen_mean':   chosen.mean(),
        'chosen_std':    chosen.std(),
        'rejected_mean': rejected.mean(),
        'rejected_std':  rejected.std(),
        'overall_mean':  all_r.mean(),
        'overall_std':   all_r.std(),
        'accuracy':      (chosen > rejected).mean(),
        'mean_gap':      (chosen - rejected).mean(),
    }

    print(f"\nReward Distribution Analysis")
    print(f"{'─'*40}")
    print(f"  Chosen   : mean={stats['chosen_mean']:+.3f}  "
          f"std={stats['chosen_std']:.3f}")
    print(f"  Rejected : mean={stats['rejected_mean']:+.3f}  "
          f"std={stats['rejected_std']:.3f}")
    print(f"  Gap      : mean={stats['mean_gap']:.3f}")
    print(f"  Accuracy : {stats['accuracy']:.2%}")

    # Health check
    if stats['overall_std'] < 0.1:
        print("  ⚠ Low variance — model may not have learned to discriminate")
    if stats['overall_std'] > 10.0:
        print("  ⚠ High variance — model may be overfit, check for length bias")
    if stats['accuracy'] < 0.6:
        print("  ⚠ Low accuracy — model is barely better than random")
    if stats['accuracy'] > 0.95:
        print("  ⚠ Very high accuracy — may be overfitting to surface features")

    return stats

### Length bias detection

[A common failure mode: the reward model learns to prefer longer responses
regardless of quality.]{.underline} This happens when the training data has a correlation
between length and human preference (humans often prefer more detailed answers,
which happens to be longer).

The downstream consequence: GRPO will optimize for verbosity rather than
quality.

In [ ]:
@torch.no_grad()
def check_length_bias(
    model:      RewardModel,
    tokenizer,
    device:     torch.device,
    prompt:     str = "Explain photosynthesis.",
    lengths:    list[int] = None,
) -> None:
    """
    Score responses of varying lengths with identical content.
    A reward model with length bias will score longer responses higher
    regardless of whether the extra length adds information.
    """
    if lengths is None:
        lengths = [20, 50, 100, 200]

    # Generate responses of controlled length by repeating a sentence
    base = "Photosynthesis is the process by which plants convert light into energy."

    model.eval()
    print("\nLength Bias Check:")
    print(f"{'─'*50}")
    print(f"  {'Length':>8}  {'Reward':>10}  Response preview")

    scores = []
    for target_words in lengths:
        # Repeat base sentence to reach target length
        words    = (base + ' ') * (target_words // len(base.split()) + 1)
        response = ' '.join(words.split()[:target_words])

        text   = (f"{SPECIAL_TOKENS['user']}\n{prompt}\n{SPECIAL_TOKENS['end']}\n"
                  f"{SPECIAL_TOKENS['assistant']}\n{response}\n{SPECIAL_TOKENS['end']}")
        tokens = tokenizer.encode(text)
        ids    = torch.tensor([tokens], dtype=torch.long, device=device)
        reward = model(ids).item()
        scores.append(reward)

        print(f"  {target_words:>8}  {reward:>10.4f}  "
              f"'{response[:40]}...'")

    # Correlation between length and score
    corr = np.corrcoef(lengths, scores)[0, 1]
    print(f"\n  Length-reward correlation: {corr:.3f}")
    if corr > 0.8:
        print("  ⚠ Strong length bias detected!")
        print("    Fix: add length-controlled pairs to training data,")
        print("         or add a length penalty to the reward.")
    elif corr > 0.5:
        print("  ⚠ Moderate length bias — monitor in downstream GRPO.")
    else:
        print("  ✓ No significant length bias.")

---

## 7. Best-of-N Sampling

Before running full GRPO, Best-of-N sampling is a simple and effective
way to use the reward model. Generate $N$ responses for each prompt,
score them all, and return the highest-scoring one.

[[[For a fixed inference budget, Best-of-N improves response quality
approximately as $\log N$]{.mark} — doubling the number of samples gives a fixed
increment in quality. It is often competitive with full RLHF at small $N$.

In [ ]:
@torch.no_grad()
def best_of_n(
    policy_model:  nn.Module,
    reward_model:  RewardModel,
    tokenizer,
    prompt:        str,
    n:             int   = 8,
    max_new_tokens: int  = 100,
    temperature:   float = 0.8,
    device:        torch.device = None,
) -> tuple[str, float, list]:
    """
    Generate N responses, score them, return the best.

    Returns:
        best_response: the highest-scoring response text
        best_score:    its reward score
        all_scored:    list of (response, score) sorted by score descending
    """
    if device is None:
        device = next(policy_model.parameters()).device

    prompt_text = (f"{SPECIAL_TOKENS['user']}\n{prompt.strip()}\n"
                   f"{SPECIAL_TOKENS['end']}\n{SPECIAL_TOKENS['assistant']}\n")
    prompt_ids  = torch.tensor(
        [tokenizer.encode(prompt_text)], dtype=torch.long, device=device
    )

    policy_model.eval()
    reward_model.eval()

    candidates = []
    for _ in range(n):
        # Sample a response from the policy
        with torch.no_grad():
            output_ids = policy_model.generate(
                prompt_ids,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                eos_token_id=tokenizer.eos_id,
            )
        # Decode only the new tokens
        response_ids  = output_ids[0, prompt_ids.size(1):]
        response_text = tokenizer.decode(response_ids.tolist())

        # Score the full (prompt + response) sequence
        full_ids = output_ids   # (1, prompt_len + response_len)
        score    = reward_model(full_ids).item()

        candidates.append((response_text, score))

    # Sort by score descending
    candidates.sort(key=lambda x: x[1], reverse=True)

    best_response, best_score = candidates[0]
    return best_response, best_score, candidates

### The diminishing returns of Best-of-N

In [ ]:
def plot_best_of_n_scaling(
    policy_model,
    reward_model,
    tokenizer,
    prompts:     list[str],
    max_n:       int = 64,
    device:      torch.device = None,
):
    """
    For a list of prompts, compute the expected best reward as a function
    of N. Plots the Best-of-N scaling curve.
    """
    import matplotlib.pyplot as plt

    n_values = [1, 2, 4, 8, 16, 32, 64]
    n_values = [n for n in n_values if n <= max_n]

    # Generate max_n candidates for each prompt once
    all_scores = []
    for prompt in prompts[:20]:   # limit to 20 prompts for speed
        _, _, candidates = best_of_n(
            policy_model, reward_model, tokenizer,
            prompt, n=max_n, device=device
        )
        scores = [s for _, s in candidates]
        all_scores.append(sorted(scores, reverse=True))

    # Best-of-N = expected maximum over N draws
    mean_best = []
    for n in n_values:
        # Average the n-th best score across prompts
        best_n = np.mean([scores[0] for scores in all_scores])
        # More precisely: best of first n for each prompt
        best_n = np.mean([max(scores[:n]) for scores in all_scores])
        mean_best.append(best_n)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(n_values, mean_best, 'o-', color='#2196F3', lw=2, ms=7)
    ax.set_xscale('log', base=2)
    ax.set_xlabel('N (number of samples)')
    ax.set_ylabel('Mean Best Reward')
    ax.set_title('Best-of-N Scaling')
    ax.set_xticks(n_values)
    ax.set_xticklabels(n_values)
    plt.tight_layout()
    plt.savefig('best_of_n_scaling.png', dpi=150)
    plt.show()
    print("Saved best_of_n_scaling.png")

---

## 8. Reward Model Calibration

A well-calibrated reward model has scores that correspond to actual
preference probabilities. A score gap of 1.0 should correspond to
$\sigma(1.0) \approx 73\%$ probability of preferring the chosen response.

In practice, reward models tend to be miscalibrated: they assign large
absolute scores (the model is overconfident) or scores that drift during
training (reward hacking). [[[Before using the RM in GRPO, normalize scores
to have zero mean and unit variance on a held-out calibration set:]{.underline}

In [ ]:
class NormalizedRewardModel(nn.Module):
    """
    Wraps a RewardModel with running normalization.
    Keeps a running mean and std of reward scores and normalizes outputs.
    This prevents reward score magnitude from changing during GRPO training,
    which would destabilize the KL penalty balance.
    """

    def __init__(self, reward_model: RewardModel, momentum: float = 0.99):
        super().__init__()
        self.rm       = reward_model
        self.momentum = momentum
        self.register_buffer('running_mean', torch.tensor(0.0))
        self.register_buffer('running_var',  torch.tensor(1.0))

    def forward(self, input_ids, attention_mask=None):
        raw = self.rm(input_ids, attention_mask)

        if self.training:
            # Update running statistics
            batch_mean = raw.mean().detach()
            batch_var  = raw.var().detach()
            self.running_mean = (self.momentum * self.running_mean
                                 + (1 - self.momentum) * batch_mean)
            self.running_var  = (self.momentum * self.running_var
                                 + (1 - self.momentum) * batch_var)

        # Normalize
        return (raw - self.running_mean) / (self.running_var.sqrt() + 1e-8)

    def calibrate(self, dataloader, device, n_batches=100):
        """
        Compute mean/std from a calibration set and set running stats.
        Call once after training the reward model, before GRPO.
        """
        self.rm.eval()
        scores = []
        with torch.no_grad():
            for i, (c_ids, c_mask, _, _) in enumerate(dataloader):
                if i >= n_batches:
                    break
                r = self.rm(c_ids.to(device), c_mask.to(device))
                scores.extend(r.cpu().tolist())

        scores = torch.tensor(scores)
        self.running_mean = scores.mean()
        self.running_var  = scores.var()
        print(f"Calibrated: mean={self.running_mean:.3f}  "
              f"std={self.running_var.sqrt():.3f}")

---

## Summary

| Concept | Key detail |
|---|---|
| Bradley-Terry model | $P(y_w \succ y_l) = \sigma(r_w - r_l)$. Logistic regression on reward gaps. |
| RM architecture | Pretrained backbone + scalar linear head. Score at last real token position. |
| [Head init to zero]{.mark} | Neutral initial rewards. Training starts from a stable, well-defined baseline. |
| Reward loss | $-\log \sigma(r_w - r_l)$. Equivalent to BCE with target=1 and logit=gap. |
| Margin loss | Requires gap > margin before loss reaches zero. Encourages confident separation. |
| Freeze early layers | Prevents overfitting backbone on small preference datasets. |
| Separate LR for head | Head learns from scratch (needs higher LR); backbone is pretrained (lower LR). |
| Length bias | RM correlates reward with length. Diagnose with controlled-length probes. |
| Best-of-N | Generate N, score all, return best. Quality scales as $\log N$. Cheap alternative to GRPO. |
| Reward normalization | Zero mean, unit variance on calibration set. Prevents score drift destabilizing GRPO. |

---

## Exercises

**1.** Verify the Bradley-Terry connection to logistic regression: construct
a simple dataset of 100 (chosen_score, rejected_score) pairs where scores
are drawn from $\mathcal{N}(1, 1)$ for chosen and $\mathcal{N}(-1, 1)$ for
rejected. Fit a logistic regression model with feature = `chosen - rejected`
and target = 1. Show that its coefficient converges to 1.0 and its
intercept to 0.0 — the reward model's implicit assumption.

**2.** Implement `compare_margin_values`: train three reward models with
`margin` ∈ {0.0, 0.5, 1.0} for 500 steps each. Plot the distribution
of reward gaps at the end of training for each. Confirm that larger
margins produce wider gap distributions and slightly lower accuracy
(harder objective, but more confident scoring).

**3.** Implement the length bias fix: add a length penalty term to the
reward loss that subtracts a small constant per token from the raw
reward score before training. Specifically:
$r_{\text{penalized}}(x, y) = r(x, y) - \lambda \cdot |y|_{\text{tokens}}$
where $\lambda$ is a small constant (try 0.001). Verify that
`check_length_bias` reports lower correlation after training with
the penalty.

**4.** Implement `reward_calibration_plot`: after training, collect reward
scores for all chosen and rejected responses in the validation set.
For each decile of reward gap, compute the empirical preference accuracy
(what fraction of pairs in that gap decile are correctly ranked). Plot
empirical accuracy vs $\sigma(\text{mean gap in decile})$. A calibrated
model should fall close to the diagonal.

**5.** Implement `reward_model_ensemble`: train three reward models with
different random seeds. For inference, average their scores. Compare the
accuracy and length-bias correlation of the ensemble vs any single model.
Ensembling is a practical technique used in real RLHF pipelines to reduce
reward hacking.

**6.** Best-of-N has a compute cost of $N$ forward passes. An alternative is
**speculative scoring**: run a small, fast reward model to filter from
$N$ down to $k$, then score those $k$ with the large reward model.
Implement this two-stage scoring pipeline and compare its quality/compute
tradeoff against standard Best-of-N at $N=16$ and $N=64$.